# Check partial-label loss gradients

Compare the legacy `diceCELoss(domain_adaptation=True)` with the version-2 `diceCELoss_gradient_fix`. The legacy objective receives softmax probabilities; the corrected objective receives raw logits and excludes unavailable anatomical logits before normalization.

Channel zero remains a softmax competitor but has no explicit Dice or CE term. Dice uses available foreground masks, including their known negatives. CE uses known foreground targets only. Each available ROI must be fully delineated throughout the scan; sparse annotation needs a separate spatial validity mask.

`-1` means unavailable, `0` an available negative, and `1` a positive target. Run all cells to check the mathematics and gradients. Native Plotly displays require no widgets or HTML exports. This notebook does not train a model or establish segmentation performance.

In [1]:
from pathlib import Path
from collections.abc import Callable
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

current_directory: Path = Path.cwd().resolve()
repository_root: Path = current_directory
for candidate in (current_directory, *current_directory.parents):
    if (candidate / "src/evaluation/losses.py").is_file():
        repository_root = candidate
        break
else:
    raise FileNotFoundError("Run this notebook inside the RNET3D repository")

sys.path.insert(0, str(repository_root))
from src.evaluation.losses import diceCELoss, diceCELoss_gradient_fix, masked_softmax_gradient_fix
from src.models.networks import mt_r_net_3d

renderer: str = "vscode"

## One partially annotated voxel

Background and channels 1, 3, 7 and 10 remain in the conditional softmax. Channel 1 is the known foreground target; the other anatomical channels are unavailable. Equal initial logits make the denominator effect easy to see.

In [2]:
label_ids: list[int] = [0, 1, 2, 3, 5, 6, 12, 13, 15, 16, 21]
target: tf.Tensor = tf.constant([[[[[0., 1., -1., 0., -1., -1., -1., 0., -1., -1., 0.]]]]])
initial_logits: np.ndarray = np.zeros(target.shape, dtype=np.float32)
available: np.ndarray = target.numpy().reshape(-1) != -1

gamma: float = 0.8
smooth: float = 1e-5
legacy_loss: Callable = diceCELoss(gamma=gamma, smooth=smooth, batch_wise=True, domain_adaptation=True)
fixed_loss: Callable = diceCELoss_gradient_fix(gamma=gamma, smooth=smooth, batch_wise=True)


def evaluate(values: np.ndarray, labels: tf.Tensor, fixed: bool) -> tuple[float, np.ndarray, np.ndarray]:
    """Return loss, probabilities and derivatives with respect to RAW logits."""
    logits: tf.Variable = tf.Variable(values)

    with tf.GradientTape() as tape:
        if fixed:
            value: tf.Tensor = fixed_loss(labels, logits)
        else:
            probabilities: tf.Tensor = tf.nn.softmax(logits, axis=-1)
            value = legacy_loss(labels, probabilities)

    gradient: tf.Tensor = tape.gradient(value, logits)
    if fixed:
        probabilities = masked_softmax_gradient_fix(labels, logits)
    else:
        probabilities = tf.nn.softmax(logits, axis=-1)

    return float(value.numpy()), probabilities.numpy(), gradient.numpy()


old_value, old_probability, old_gradient = evaluate(initial_logits, target, fixed=False)
new_value, new_probability, new_gradient = evaluate(initial_logits, target, fixed=True)
comparison: pd.DataFrame = pd.DataFrame({
    "channel": np.arange(len(label_ids)),
    "original label ID": label_ids,
    "available": available,
    "old probability": old_probability.reshape(-1),
    "conditional probability": new_probability.reshape(-1),
    "old dL/dlogit": old_gradient.reshape(-1),
    "fixed dL/dlogit": new_gradient.reshape(-1),
})
display(comparison)
print(f"Old loss: {old_value:.8f}; fixed loss: {new_value:.8f}")

np.testing.assert_array_equal(new_gradient.reshape(-1)[~available], 0)
assert np.any(np.abs(old_gradient.reshape(-1)[~available]) > 1e-5)
assert np.any(np.abs(new_gradient.reshape(-1)[available]) > 1e-5)
assert new_gradient[..., 0].item() > 0  # Background competes at the known foreground voxel.

figure: go.Figure = go.Figure()
figure.add_bar(x=comparison["channel"], y=comparison["old dL/dlogit"], name="Original")
figure.add_bar(x=comparison["channel"], y=comparison["fixed dL/dlogit"], name="Gradient fix")
figure.update_layout(title="Logit gradients: background plus channels 1, 3, 7, 10 are available", xaxis_title="Channel", yaxis_title="dL/dlogit", barmode="group")
figure.show(renderer=renderer)

,channel,original label ID,available,old probability,conditional probability,old dL/dlogit,fixed dL/dlogit
0,0,0,True,0.090909,0.2,0.006867,0.042776
1,1,1,True,0.090909,0.2,-0.068642,-0.171112
2,2,2,False,0.090909,0.0,0.006863,0.000000
3,3,3,True,0.090909,0.2,0.006867,0.042779
4,4,5,False,0.090909,0.0,0.006863,0.000000
5,5,6,False,0.090909,0.0,0.006863,0.000000
6,6,12,False,0.090909,0.0,0.006863,0.000000
7,7,13,True,0.090909,0.2,0.006867,0.042779
8,8,15,False,0.090909,0.0,0.006863,0.000000
9,9,16,False,0.090909,0.0,0.006863,0.000000


Old loss: 0.15316395; fixed loss: 0.50521314


## Change an unavailable logit

The original valid probability and loss change through the full softmax denominator. The corrected conditional probabilities, loss and gradients remain unchanged. Absolute old/new loss values are not a quality comparison: version 2 changes foreground supervision, smoothing and class reductions.

In [3]:
rows: list[dict] = []
for ignored_logit in np.linspace(-8, 8, 33):
    changed: np.ndarray = initial_logits.copy()
    changed[..., 2] = ignored_logit
    old_loss_value, old_prob, _ = evaluate(changed, target, fixed=False)
    fixed_loss_value, fixed_prob, _ = evaluate(changed, target, fixed=True)
    rows.append({"ignored logit": ignored_logit, "old loss": old_loss_value, "fixed loss": fixed_loss_value,
                 "old target probability": old_prob[..., 1].item(), "fixed target probability": fixed_prob[..., 1].item()})

sweep: pd.DataFrame = pd.DataFrame(rows)
np.testing.assert_array_equal(sweep["fixed loss"].to_numpy(), new_value)
expected_probability: float = 1 / int(available.sum())
np.testing.assert_allclose(sweep["fixed target probability"].to_numpy(), expected_probability, atol=1e-7)

figure = make_subplots(rows=1, cols=2, subplot_titles=("Loss", "Probability of known target"))
for prefix in ("old", "fixed"):
    loss_trace = go.Scatter(x=sweep["ignored logit"], y=sweep[f"{prefix} loss"], name=prefix)
    probability_trace = go.Scatter(x=sweep["ignored logit"], y=sweep[f"{prefix} target probability"], name=prefix, showlegend=False)
    figure.add_trace(loss_trace, row=1, col=1)
    figure.add_trace(probability_trace, row=1, col=2)
figure.update_xaxes(title_text="Unavailable logit (channel 2)")
figure.show(renderer=renderer)

for extreme in (-10000., 10000.):
    changed = initial_logits.copy()
    changed[..., 2] = extreme
    value, _, gradient = evaluate(changed, target, fixed=True)
    assert value == new_value
    np.testing.assert_array_equal(gradient.reshape(-1)[~available], 0)
print("Unavailable logits have no effect, including extreme values.")

Unavailable logits have no effect, including extreme values.


## Independent finite-difference check

Compare automatic differentiation with a central finite difference for one available and one unavailable logit.

In [4]:
epsilon: float = 1e-3
checks: list[dict] = []
for fixed in (False, True):
    _, _, analytical = evaluate(initial_logits, target, fixed=fixed)
    for channel in (1, 2):
        plus = initial_logits.copy()
        minus = initial_logits.copy()
        plus[..., channel] += epsilon
        minus[..., channel] -= epsilon
        upper, _, _ = evaluate(plus, target, fixed=fixed)
        lower, _, _ = evaluate(minus, target, fixed=fixed)
        numerical: float = (upper - lower) / (2 * epsilon)
        derivative: float = analytical[..., channel].item()
        np.testing.assert_allclose(numerical, derivative, atol=5e-5, rtol=0.01)
        checks.append({"fixed": fixed, "channel": channel, "autodiff": derivative, "finite difference": numerical})
display(pd.DataFrame(checks))

,fixed,channel,autodiff,finite difference
0,False,1,-0.068642,-0.068650
1,False,2,0.006863,0.006862
2,True,1,-0.171112,-0.171095
3,True,2,0.000000,0.000000


## Foreground CE and background competition

Stored background/ambiguous zeros have no explicit CE term. An available foreground mask still provides Dice negatives at those voxels. Background can receive gradients through competition, whereas unavailable anatomical logits receive exactly zero ROI-loss gradient. Annotated empty ROIs remain supervised by Dice; background-only or entirely unavailable targets contribute zero when no foreground ROI is annotated.

In [5]:
ce_only: Callable = diceCELoss_gradient_fix(gamma=1.0, smooth=smooth, batch_wise=True)
dice_only: Callable = diceCELoss_gradient_fix(gamma=0.0, smooth=smooth, batch_wise=True)

cases: dict[str, list[float]] = {
    "Known foreground": [0., 1., -1.],
    "Zero / task remainder": [1., 0., -1.],
    "Known ROI negative, unspecified anatomy": [0., 0., -1.],
    "No foreground annotation": [1., -1., -1.],
    "Entirely unavailable": [-1., -1., -1.],
}
case_rows: list[dict] = []
for name, entries in cases.items():
    labels = tf.constant([[[[entries]]]])
    values = np.zeros(labels.shape, dtype=np.float32)
    total_value, _, gradient = evaluate(values, labels, fixed=True)
    ce_value = float(ce_only(labels, values).numpy())
    dice_value = float(dice_only(labels, values).numpy())
    case_rows.append({
        "Case": name, "CE": ce_value, "Dice loss": dice_value,
        "Combined loss": total_value, "Background gradient": gradient[..., 0].item(),
        "Foreground gradient": gradient[..., 1].item(),
        "Unavailable gradient": gradient[..., 2].item(),
    })
    assert gradient[..., 2].item() == 0
    if name != "Known foreground":
        assert ce_value == 0
    if name in ("No foreground annotation", "Entirely unavailable"):
        assert total_value == 0
        np.testing.assert_array_equal(gradient, 0)

display(pd.DataFrame(case_rows))
print("PASS: foreground-only CE, background competition, annotated empty ROIs, and zero unavailable-logit gradients.")

,Case,CE,Dice loss,Combined loss,Background gradient,Foreground gradient,Unavailable gradient
0,Known foreground,0.693147,0.333331,0.621184,0.444444,-0.444444,0.0
1,Zero / task remainder,0.000000,0.999980,0.199996,-0.000002,0.000002,0.0
2,"Known ROI negative, unspecified anatomy",0.000000,0.999980,0.199996,-0.000002,0.000002,0.0
3,No foreground annotation,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
4,Entirely unavailable,0.000000,0.000000,0.000000,0.000000,0.000000,0.0


PASS: foreground-only CE, background competition, annotated empty ROIs, and zero unavailable-logit gradients.


## False positives outside an annotated ROI

The second voxel is zero in the mask, but remains a known negative for the fully annotated ROI. Increase its foreground prediction while keeping the positive voxel unchanged. Dice must penalize that false positive, while foreground-only CE remains unchanged.

In [6]:
negative_target = tf.constant([[[[[0., 1., -1.], [1., 0., -1.]]]]])
clean_logits = np.array([[[[[0., 2., 100.], [2., 0., 100.]]]]], dtype=np.float32)
false_positive_logits = np.array([[[[[0., 2., 100.], [0., 2., 100.]]]]], dtype=np.float32)

clean_dice = float(dice_only(negative_target, clean_logits).numpy())
false_logits = tf.Variable(false_positive_logits)
with tf.GradientTape() as tape:
    false_dice_value = dice_only(negative_target, false_logits)
false_gradient = tape.gradient(false_dice_value, false_logits).numpy()
false_dice = float(false_dice_value.numpy())
clean_ce = float(ce_only(negative_target, clean_logits).numpy())
false_ce = float(ce_only(negative_target, false_positive_logits).numpy())

assert false_dice > clean_dice
assert false_gradient[0, 0, 0, 1, 1] > 0
assert false_gradient[0, 0, 0, 1, 0] < 0
np.testing.assert_array_equal(false_gradient[..., 2], 0)
np.testing.assert_allclose(false_ce, clean_ce)

negative_comparison = pd.DataFrame({
    "Prediction": ["Correct negative", "False positive"],
    "Foreground Dice loss": [clean_dice, false_dice],
    "Foreground CE": [clean_ce, false_ce],
})
display(negative_comparison)
negative_figure = go.Figure()
negative_figure.add_bar(x=negative_comparison["Prediction"], y=negative_comparison["Foreground Dice loss"], name="Dice loss")
negative_figure.update_layout(title="Known ROI negatives retain false-positive supervision", yaxis_title="Foreground Dice loss")
negative_figure.show(renderer=renderer)
print("PASS: Dice pushes the false-positive ROI logit down and the background competitor up; unavailable logits remain isolated.")

,Prediction,Foreground Dice loss,Foreground CE
0,Correct negative,0.119202,0.126928
1,False positive,0.362108,0.126928


PASS: Dice pushes the false-positive ROI logit down and the background competitor up; unavailable logits remain isolated.


## Mixed annotations and graph execution

A class unavailable in one sample can be supervised in another. All foreground classes unavailable gives zero loss and gradient. Check the same masking behavior inside `tf.function`. Complete masks use the same foreground-only objective; equivalence with the legacy background-inclusive loss is not expected.

In [7]:
mixed_target = tf.constant([[[[[0., 1., -1.]]]], [[[[0., 0., 1.]]]]])
mixed_logits = np.zeros(mixed_target.shape, dtype=np.float32)
_, _, mixed_gradient = evaluate(mixed_logits, mixed_target, fixed=True)
assert mixed_gradient[0, 0, 0, 0, 2] == 0
assert abs(mixed_gradient[1, 0, 0, 0, 2]) > 1e-5

graph_loss = tf.function(fixed_loss)
graph_logits = tf.Variable(initial_logits)
with tf.GradientTape() as tape:
    graph_value = graph_loss(target, graph_logits)
graph_gradient = tape.gradient(graph_value, graph_logits).numpy()
np.testing.assert_allclose(graph_value.numpy(), new_value, atol=1e-7)
np.testing.assert_array_equal(graph_gradient.reshape(-1)[~available], 0)

all_ignored = -tf.ones_like(target)
with tf.GradientTape() as tape:
    ignored_value = graph_loss(all_ignored, graph_logits)
ignored_gradient = tape.gradient(ignored_value, graph_logits).numpy()
assert float(ignored_value.numpy()) == 0
np.testing.assert_array_equal(ignored_gradient, 0)
print("PASS: mixed annotation availability, unavailable batches, and graph-mode gradients.")

PASS: mixed annotation availability, unavailable batches, and graph-mode gradients.


## Use logits from the existing network

The inference builder still exposes `regions` softmax probabilities. The fitting view exposes `regions_logits`, sharing the same weights and retaining the brain/classifier heads. Do not attach the corrected loss directly to the probability output.

Zero-gradient statements refer to the ROI loss derivative with respect to unavailable ROI logits. Other samples and auxiliary tasks can still update shared network weights.

In [8]:
spatial_shape: tuple[int, int, int] = (16, 16, 16)
filter_widths: list[int] = [1, 2, 4, 8, 16]
inference_model: tf.keras.Model = mt_r_net_3d(spatial_shape, 1, 11, filter_widths, with_classifier=True)
regions_layer = inference_model.get_layer("regions")
assert regions_layer.get_config()["activation"] == "softmax"

linear_regions = tf.keras.layers.Activation("linear", dtype="float32", name="regions_logits")
roi_logits = linear_regions(regions_layer.input)
training_outputs = [roi_logits, inference_model.get_layer("brain_mask").output, inference_model.get_layer("classifier").output]
training_model = tf.keras.Model(inference_model.inputs, training_outputs, name="rnet_gradient_fix")

inputs = tf.zeros((1, 16, 16, 16, 1), dtype=tf.float32)
inference_outputs = inference_model(inputs, training=False)
logit_outputs = training_model(inputs, training=False)
restored_probabilities = tf.nn.softmax(logit_outputs[0], axis=-1)
np.testing.assert_allclose(restored_probabilities.numpy(), inference_outputs[0].numpy(), atol=1e-7)
assert training_model.output_names == ["regions_logits", "brain_mask", "classifier"]
assert {id(weight) for weight in training_model.weights} == {id(weight) for weight in inference_model.weights}

volume_target = tf.tile(target, [1, 16, 16, 16, 1])
with tf.GradientTape() as tape:
    outputs = training_model(inputs, training=False)
    roi_value = fixed_loss(volume_target, outputs[0])
roi_gradient = tape.gradient(roi_value, outputs[0]).numpy()
np.testing.assert_array_equal(roi_gradient[..., ~available], 0)
assert np.any(np.abs(roi_gradient[..., available]) > 0)
print("PASS: unchanged inference probabilities, shared weights, and zero ignored-logit gradients in the real network.")
print("Training outputs:", training_model.output_names)

PASS: unchanged inference probabilities, shared weights, and zero ignored-logit gradients in the real network.
Training outputs: ['regions_logits', 'brain_mask', 'classifier']


Version 2 averages foreground Dice with `(2 * intersection + smooth) / (union + smooth)`. CE/focal averages over available foreground entries at foreground-positive voxels, retaining class-mean scaling. It intentionally differs from the legacy loss even for complete masks.

New MouseFull runs record `LossConfig(regions="dice_ce_gradient_fix", gradient_fix_version=2)`. Older version-1 runs can be evaluated or used for initialization, but cannot silently resume with the changed objective. The inference architecture remains unchanged. Retraining and full-volume evaluation are needed to assess the corrected method.

## V3: verified lesion-free animals and saturated false positives

A verified SHAM scan has an **available empty lesion target (0)**, not an
unavailable lesion target (-1). V3 adds balanced positive/negative foreground BCE
to foreground Dice and computes the binary terms stably from logits. This cell
shows a strong corrective gradient when a lesion false positive is nearly certain.
The unavailable anatomical logit stays excluded, even when it is very large.
These checks establish gradient behavior, not segmentation performance.


In [ ]:
healthy_target: tf.Tensor = tf.constant([[[[[1.0, 0.0, -1.0]]]]])
healthy_values: tf.Tensor = tf.constant([[[[[0.0, 20.0, 1000.0]]]]])
healthy_rows: list[dict[str, float | int]] = []

for loss_version in (1, 2, 3):
    objective = diceCELoss_gradient_fix(
        gamma=0.8, batch_wise=True, version=loss_version,
    )
    healthy_logits = tf.Variable(healthy_values)
    with tf.GradientTape() as tape:
        healthy_loss = objective(healthy_target, healthy_logits)
    healthy_gradient = tape.gradient(healthy_loss, healthy_logits).numpy().reshape(-1)

    assert healthy_gradient[2] == 0.0
    if loss_version == 3:
        assert healthy_gradient[1] > 0.5

    healthy_rows.append({
        "Version": loss_version, "Loss": float(healthy_loss.numpy()),
        "Background gradient": float(healthy_gradient[0]),
        "Lesion gradient": float(healthy_gradient[1]),
        "Unavailable anatomy gradient": float(healthy_gradient[2]),
    })

healthy_comparison = pd.DataFrame(healthy_rows)
display(healthy_comparison)
